<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/chgnet_mlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install chgnet

In [ ]:
import os
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -e .

In [ ]:
!wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip

In [ ]:
import json,zipfile
mlearn = json.loads(
        zipfile.ZipFile("mlearn.json.zip").read(
            "mlearn.json"
        )
    )

In [ ]:
import os
from jarvis.core.atoms import Atoms
os.chdir('/content')
elements = ['Si']
for element in elements:

    benchmark_energies = (
        "jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/mlearn_"
        + element
        + "_energy.json.zip"
    )
    temp_energies = benchmark_energies.split("/")[-1].split(".zip")[0]
    energies = json.loads(
        zipfile.ZipFile(benchmark_energies).read(temp_energies)
    )
    train_ids = list(energies["train"].keys())
    test_ids = list(energies["test"].keys())

    train_energies=[]
    train_forces=[]
    train_stresses=[]
    train_structures=[]
    for i in mlearn:
        if i["jid"] in train_ids:
            # print(i)
            train_energies.append(i["energy"])
            train_forces.append(i["forces"])
            train_stresses.append(i['stresses'])
            atoms = Atoms.from_dict(i["atoms"])
            train_structures.append(atoms.pymatgen_converter())



In [ ]:
pip install numpy==1.23.5

In [ ]:
pip install -U ase

In [ ]:
from chgnet.data.dataset import StructureData, get_train_val_test_loader
from chgnet.trainer import Trainer
from chgnet.model.model import CHGNet
dataset = StructureData(
    structures=train_structures,
    energies=train_energies,
    forces=train_forces,
    stresses=train_stresses,
    #magmoms=list_of_magmoms,
)
train_loader, val_loader, test_loader = get_train_val_test_loader(
    dataset, batch_size=32, train_ratio=0.9, val_ratio=0.05
)
trainer = Trainer(
    model=CHGNet(),
    targets="efsm",
    optimizer="Adam",
    criterion="MSE",
    learning_rate=1e-2,
    epochs=50,
    use_device="cuda",
)

trainer.train(train_loader, val_loader, test_loader)